# Heterogeneous-array mosaic simulation (ALMA 7 m + 12 m)

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/simulation/heterogeneous_array_mosaic_simulation.ipynb)

Port of the SIRIUS `heterogeneous_array_mosaic_simulations` notebook: a mixed array of ALMA
7 m (ACA) and 12 m antennas observing a two-field mosaic, with and without thermal noise.  The
analysis that used CASA tasks (``listobs``, ``visstat``, ``tclean``) is done here directly on the
MSv4 processing set with xarray, and the fields are imaged one at a time with AstroVIPER.

---

## Background

- Each antenna gets its own beam model through ``beam_model_map`` (here an ACA and an ALMA Airy disk);
  the primary beam of a baseline is the product of the two antenna beams.
- A **mosaic** is a time-varying phase centre: ``phase_center_ra_dec`` has one row per time and
  ``field_name`` labels the fields.  All fields live in **one** MSv4 (``field_name`` is a
  coordinate along ``time``) as in real MSv4 data.
- Thermal noise follows the ``tsys-manual`` model of ``casatools.simulator.setnoise``; the weights
  scale with the product of the two dish areas.

## API


In [ ]:
from astroviper.distributed_applications.simulation import simulate_processing_set

simulate_processing_set?

## Install AstroVIPER

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401

    print("Using astroviper version", version("astroviper"))
except ImportError:
    os.system("pip install --upgrade astroviper")
    import astroviper  # noqa: F401

    print("Installed astroviper version", version("astroviper"))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from astropy.coordinates import SkyCoord

xr.set_options(display_style="html")
ARCSEC_TO_RAD = np.pi / (180 * 3600)

In [ ]:
from toolviper.dask.client import local_client

viper_client = local_client(cores=4, memory_limit="4GB")
viper_client

## Heterogeneous array: 4 x 7 m + 8 x 12 m antennas

In [ ]:
from astroviper.utils.telescope_layout import read_telescope_layout

alma_all = read_telescope_layout("alma.all")
dish = alma_all.ANTENNA_DISH_DIAMETER.values
selection = np.concatenate(
    [np.where(dish == 7.0)[0][:4], np.where(dish == 12.0)[0][:8]]
)
antenna_xds = alma_all.isel(antenna_name=selection)
n_antenna = antenna_xds.sizes["antenna_name"]
print(list(antenna_xds.antenna_name.values))
print(antenna_xds.ANTENNA_DISH_DIAMETER.values)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
position = (
    antenna_xds.ANTENNA_POSITION.values
    - antenna_xds.ANTENNA_POSITION.values.mean(axis=0)
)
for diameter, color in [(7.0, "tab:red"), (12.0, "tab:blue")]:
    sel = antenna_xds.ANTENNA_DISH_DIAMETER.values == diameter
    ax.scatter(position[sel, 0], position[sel, 1], c=color, label=f"{diameter:.0f} m")
for name, pos in zip(antenna_xds.antenna_name.values, position, strict=True):
    ax.annotate(name, pos[:2])
ax.set_xlabel("x offset [m]")
ax.set_ylabel("y offset [m]")
ax.legend()
ax.set_aspect("equal")
plt.show()

## Beam models per antenna

In [ ]:
from astroviper.utils.beam_models import airy_disk_model

beam_models = [airy_disk_model("aca"), airy_disk_model("alma")]
beam_model_map = np.where(antenna_xds.ANTENNA_DISH_DIAMETER.values == 7.0, 0, 1)
print(beam_models)
print(beam_model_map)

## Two-field mosaic

In [ ]:
n_time = 18
time_params = {
    "time_start": "2019-10-03T19:00:00.000",
    "time_delta": 2000.0,
    "n_samples": n_time,
}
frequency_params = {
    "freq_start": 90e9,
    "freq_delta": 0.5e9,
    "n_channels": 5,
    "channel_width": 0.5e9,
    "spectral_window_name": "Band3",
}
polarization = ["XX", "YY"]

field_1 = SkyCoord(ra="19h59m28.5s", dec="-40d44m01.5s", frame="icrs")
field_2 = SkyCoord(ra="19h59m28.5s", dec="-40d44m51.5s", frame="icrs")
half = n_time // 2
phase_center_ra_dec = np.array(
    [[field_1.ra.rad, field_1.dec.rad]] * half
    + [[field_2.ra.rad, field_2.dec.rad]] * half
)
field_name = ["field1"] * half + ["field2"] * half

from astroviper.utils.coordinate_transforms import sin_pixel_to_celestial_coord

image_size = np.array([256, 256])
cell_size = np.array([-1.0, 1.0]) * ARCSEC_TO_RAD
# one source per field, 25 arcsec from each field centre towards the other field
point_source_ra_dec = sin_pixel_to_celestial_coord(
    np.array([field_1.ra.rad, field_1.dec.rad]),
    image_size,
    cell_size,
    np.array([[128, 103], [128, 78]]),
)[None, :, :]
point_source_flux = np.array([[1.0, 0, 0, 1.0], [0.5, 0, 0, 0.5]])[:, None, None, :]

## Simulate without and with noise

In [ ]:
common = dict(
    antenna_xds=antenna_xds,
    time_params=time_params,
    frequency_params=frequency_params,
    polarization=polarization,
    point_source_flux=point_source_flux,
    point_source_ra_dec=point_source_ra_dec,
    phase_center_ra_dec=phase_center_ra_dec,
    field_name=field_name,
    beam_models=beam_models,
    beam_model_map=beam_model_map,
    n_time_chunks=2,
    n_frequency_chunks=5,
    overwrite=True,
)
result = simulate_processing_set(
    ps_store="het_mosaic_sim.ps.zarr", noise_params=None, **common
)
# a large receiver temperature makes the noise visible in the plots
result_noisy = simulate_processing_set(
    ps_store="het_mosaic_sim_noisy.ps.zarr",
    noise_params={"t_receiver": 5000.0, "random_seed": 1},
    **common,
)

## Validate the processing set against the MSv4 schema

``xradio.schema.check.check_datatree`` checks every dataset of the processing set (coordinates,
dimensions, dtypes and attributes of the main, antenna and field/source datasets) against the
MSv4 schema.  ``simulate_processing_set`` runs this check itself (``check_schema=True``) and logs
a warning on problems; here it is run explicitly so that the result is visible.

In [ ]:
from xradio.measurement_set import open_processing_set
from xradio.schema.check import check_datatree

ps_xdt = open_processing_set("het_mosaic_sim.ps.zarr")
issues = check_datatree(ps_xdt)
print(issues)
assert str(issues) == "No schema issues found"
# the same check works on a single MSv4 (the checker dispatches on the ``type`` attribute)
print(check_datatree(ps_xdt[result["ms_name"]]))

In [ ]:
print(check_datatree(open_processing_set("het_mosaic_sim_noisy.ps.zarr")))

## Inspect the processing set

``summary()`` replaces CASA ``listobs``; the MSv4 carries both fields.

In [ ]:
display(ps_xdt.xr_ps.summary())
ms_xds = ps_xdt[result["ms_name"]].ds
ps_xdt[result["ms_name"]]["field_and_source_base_xds"].ds

## Baseline types

Baselines are grouped into 7 m - 7 m, 12 m - 12 m and cross (7 m - 12 m).

In [ ]:
dish_by_name = dict(
    zip(
        antenna_xds.antenna_name.values,
        antenna_xds.ANTENNA_DISH_DIAMETER.values,
        strict=True,
    )
)
d1 = np.array([dish_by_name[a] for a in ms_xds.baseline_antenna1_name.values])
d2 = np.array([dish_by_name[a] for a in ms_xds.baseline_antenna2_name.values])
baseline_type = np.where(
    (d1 == 7) & (d2 == 7),
    "7m-7m",
    np.where((d1 == 12) & (d2 == 12), "12m-12m", "cross"),
)
colors = {"7m-7m": "tab:red", "12m-12m": "tab:blue", "cross": "tab:purple"}

noisy_xds = open_processing_set("het_mosaic_sim_noisy.ps.zarr")[
    result_noisy["ms_name"]
].ds
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
uvw = ms_xds.UVW.values
for btype, color in colors.items():
    sel = baseline_type == btype
    axes[0].scatter(
        uvw[:, sel, 0].ravel(), uvw[:, sel, 1].ravel(), s=2, c=color, label=btype
    )
    axes[0].scatter(-uvw[:, sel, 0].ravel(), -uvw[:, sel, 1].ravel(), s=2, c=color)
    hours = (ms_xds.time.values - ms_xds.time.values[0]) / 3600
    axes[1].plot(
        hours,
        np.abs(ms_xds.VISIBILITY.values[:, sel, 0, 0]),
        ".",
        c=color,
        ms=3,
        label=btype,
    )
    axes[2].plot(
        hours,
        np.abs(noisy_xds.VISIBILITY.values[:, sel, 0, 0]),
        ".",
        c=color,
        ms=3,
        label=btype,
    )
axes[0].set_xlabel("u [m]")
axes[0].set_ylabel("v [m]")
axes[0].set_title("uv coverage")
axes[0].legend()
axes[0].set_aspect("equal")
axes[1].set_xlabel("time [h]")
axes[1].set_ylabel("|V| [Jy]")
axes[1].set_title("amplitude vs time (noiseless)")
axes[2].set_xlabel("time [h]")
axes[2].set_ylabel("|V| [Jy]")
axes[2].set_title("amplitude vs time (noisy)")
plt.show()

## Expected amplitude and weight ratios

With Airy beams the primary beam of a 7 m - 12 m baseline is the geometric mean of the two
single-dish primary beams, and the thermal noise scales with $1/(D_1 D_2)$ so that the weights
scale with $(D_1 D_2)^2$.  The table below compares the measured ratios with the expectation
(the replacement of the SIRIUS ``check_vals`` table).

In [ ]:
import pandas as pd

rows = []
weight = noisy_xds.WEIGHT.values
for btype in ["7m-7m", "cross", "12m-12m"]:
    sel = baseline_type == btype
    rows.append(
        {
            "baseline type": btype,
            "mean weight": weight[:, sel].mean(),
            "weight ratio to 7m-7m": weight[:, sel].mean()
            / weight[:, baseline_type == "7m-7m"].mean(),
            "noise rms (noisy - noiseless)": (
                noisy_xds.VISIBILITY.values[:, sel] - ms_xds.VISIBILITY.values[:, sel]
            ).real.std(),
        }
    )
table = pd.DataFrame(rows)
table["expected weight ratio (D1 D2)^2 / 49^2"] = [
    1.0,
    (7 * 12) ** 2 / 49**2,
    (12 * 12) ** 2 / 49**2,
]
table

## Image each field

AstroVIPER's single-field imager images one phase centre; a joint mosaic gridder is not yet
available.  To look at the two pointings separately we simulate the same sky once per field
(the same sources, antennas and beams) and image each processing set.

In [ ]:
from xradio.image import load_image
from xradio.measurement_set import open_processing_set

from astroviper.distributed_applications.imaging import image_cube_single_field


def image_simulation(
    ps_store,
    image_store,
    image_size,
    cell_size_arcsec,
    niter=0,
    polarization_coords=("I",),
    n_chunks=2,
):
    """Make a (dirty or cleaned) cube of a simulated processing set with AstroVIPER."""
    ps_xdt = open_processing_set(ps_store)
    combined = ps_xdt.xr_ps.get_combined_field_and_source_xds()
    phase_direction = combined.FIELD_PHASE_CENTER_DIRECTION.sel(
        field_name=combined.attrs["center_field_name"]
    ).values
    image_params = {
        "image_size": list(image_size),
        "cell_size": np.array([-cell_size_arcsec, cell_size_arcsec]) * ARCSEC_TO_RAD,
        "phase_direction": phase_direction,
        "frequency_coords": ps_xdt.xr_ps.get_freq_axis().values,
        "polarization_coords": list(polarization_coords),
        "time_coords": [0],
        "fft_padding": 1.2,
        "cpp_gridder": True,
    }
    iteration_control = {
        "niter": niter,
        "nmajor": -1 if niter > 0 else 0,
        "threshold": 0.0,
        "gain": 0.1,
        "cyclefactor": 1.5,
        "cycleniter": -1,
        "minpsffraction": 0.05,
        "maxpsffraction": 0.8,
        "primary_beam_limit": 0.1,
    }
    keep = [
        "sky_residual",
        "point_spread_function",
        "primary_beam",
        "beam_fit_params_point_spread_function",
    ]
    if niter > 0:
        keep += ["sky_model", "mask"]
    image_cube_single_field(
        ps_store=ps_store,
        image_store=image_store,
        image_params=image_params,
        imaging_weights_params={
            "weighting": "natural",
            "robust": 0.5,
            "casa_weighting_implementation": True,
        },
        iteration_control_params=iteration_control,
        gridder="prolate_spheroidal",
        deconvolver="hogbom_many_threads",
        scan_intents="OBSERVE_TARGET#ON_SOURCE",
        image_data_variables_keep=keep,
        processing_set_data_group_name="base",
        single_precision_image=False,
        processing_function_threads=1,
        n_chunks=n_chunks,
        overwrite=True,
        restore=niter > 0,
    )
    return load_image(image_store)


def show_image(
    img_xds, variable="SKY_RESIDUAL", frequency=0, polarization=0, title=None, vmax=None
):
    """Plot one plane of an AstroVIPER image with l/m in arcsec."""
    plane = (
        img_xds[variable]
        .isel(time=0, frequency=frequency, polarization=polarization)
        .values
    )
    extent = (
        np.array(
            [
                img_xds.l.values[0],
                img_xds.l.values[-1],
                img_xds.m.values[0],
                img_xds.m.values[-1],
            ]
        )
        / ARCSEC_TO_RAD
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(plane.T, origin="lower", extent=extent, cmap="viridis", vmax=vmax)
    ax.set_xlabel("l [arcsec]")
    ax.set_ylabel("m [arcsec]")
    ax.set_title(
        title
        or f"{variable} channel {frequency} ({img_xds.frequency.values[frequency] / 1e9:.3f} GHz)"
    )
    fig.colorbar(im, ax=ax, label="Jy/beam")
    return fig

In [ ]:
for field, centre in [("field1", field_1), ("field2", field_2)]:
    per_field = dict(common)
    per_field.update(
        phase_center_ra_dec=np.array([[centre.ra.rad, centre.dec.rad]]),
        field_name=field,
        time_params={
            "time_start": "2019-10-03T19:00:00.000",
            "time_delta": 2000.0,
            "n_samples": half,
        },
    )
    result_field = simulate_processing_set(
        ps_store=f"het_{field}.ps.zarr", noise_params=None, **per_field
    )
    img = image_simulation(
        f"het_{field}.ps.zarr",
        f"het_{field}.img.zarr",
        image_size,
        1.0,
        niter=500,
        n_chunks=5,
    )
    show_image(
        img,
        variable="SKY_RESTORED",
        title=f"{field}: restored image (7 m, 12 m and cross baselines)",
    )
    plt.show()

## Clean up

In [ ]:
import glob
import shutil

for path in glob.glob("het_*.ps.zarr") + glob.glob("het_*.img.zarr"):
    shutil.rmtree(path, ignore_errors=True)
viper_client.close()